<a href="https://colab.research.google.com/github/elfantasies/AI/blob/main/0709_Colab_LINE_Bot_with_GEMINI_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.9/818.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 7.5 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://sequential-dispensational-tripp.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://sequential-dispensational-tripp.ngrok-free.dev


True

In [6]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [7]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [8]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology, MUST）是一所位於台灣新竹縣新豐鄉的私立科技大學，其前身為1966年創立的「明新工業專科學校」。1997年改制為「明新技術學院」，並於2002年升格為科技大學，定名為「明新科技大學」。

學校秉持「堅毅、求新、創造」的校訓精神，其校名「明新」取自《大學》「在明明德，在新民，在止於至善」之精義，旨在培養學生具備高尚品德、專業學問與優良技術，以成為「一流產業大學」為發展願景，並達成培育「跨域整合、務實創新、全人學習」之專業人才的教育目標。

明新科技大學現設有六個學院，包括半導體學院、工程學院、管理學院、民生學院、人文與設計學院以及共同教育學院，涵蓋20個學系、2個學位學程（含1個博士學位學程）及11個碩士班，學生總數逾萬人。

學校坐擁新竹科學園區與新竹工業區的豐富產業資源，因此將「產業大學」定位為辦學方向，致力於強化產學鏈結，透過契合式人才課程，使畢業生具備「立即就業」的能力。明新科大在企業界中享有良好聲譽，曾於2022年《遠見》雜誌的「企業最愛公私立技職科大調查」中，在四大領域獲得兩項「企業最愛」，起薪排名私立科大第一，甚至優於部分國立科大。在1111人力銀行統計的半導體產業界最愛聘用畢業生中，明新科大也名列第四，與台灣頂尖大學齊名，是唯一入榜的私立科大，凸顯其育才成果備受企業青睞，歸功於精準的「產官學規劃」。

此外，明新科技大學也積極推動跨國人才培育，除原有與菲律賓大學的雙聯學位外，更與澳洲西雪梨大學、越南胡志明市經濟大學簽訂合作，開啟跨國雙聯學位教育模式，以提升學生的國際移動力與全球學習視野，培養具高競爭力的跨國人才。為配合產業趨勢，學校發展MUST四大育才特色，包含多元學習（Multidisciplinary Learning）、全球視野（Universal Perspective）、永續經營（Sustainable Operations）與技術創新（Technological Innovation），引導學生跨域學習，以鎖定半導體、AI、元宇宙、風電綠能等前瞻產業，培育產業所需人才。 學校也積極建置「永續智慧商務」教學與實習場域，透過生成式AI與AI專案應用，發展智慧零售、智慧金融、智慧製造與智慧商業等實驗室，並推動「生成式產學合作模式」與「產業

In [9]:
result2 = stateful_query("校長是誰？")
print(result2)

None


In [10]:
from flask import Flask, request, abort
import logging
import os
import time
from google.genai import types

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    MessagingApiBlob,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
    FileMessageContent
)

app = Flask(__name__)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
app.logger.setLevel(logging.INFO)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

# 儲存檔案的目錄
UPLOAD_DIR = "/content/uploaded_files"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 儲存每個使用者的對話 session 和上傳的檔案
user_sessions = {}  # {user_id: {"chat": chat_object, "uploaded_file": gemini_file}}

def get_user_session(user_id):
    """取得或建立使用者的對話 session"""
    if user_id not in user_sessions:
        # 建立新的對話 session
        new_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=GenerateContentConfig(
                system_instruction="你是一個中文的AI助手，請用繁體中文回答。如果使用者有提供參考文件，請根據文件內容回答問題。",
                tools=[google_search_tool],
                response_modalities=["TEXT"],
            )
        )
        user_sessions[user_id] = {
            "chat": new_chat,
            "uploaded_file": None
        }
    return user_sessions[user_id]

def download_line_file(message_id, file_name):
    """從 LINE 下載使用者上傳的檔案"""
    with ApiClient(configuration) as api_client:
        line_bot_blob_api = MessagingApiBlob(api_client)
        file_content = line_bot_blob_api.get_message_content(message_id)

        file_path = os.path.join(UPLOAD_DIR, file_name)

        with open(file_path, 'wb') as f:
            f.write(file_content)

        return file_path

def upload_file_to_gemini(file_path):
    """上傳檔案到 Gemini Files API"""
    uploaded_file = client.files.upload(
        file=file_path,
        config={'display_name': os.path.basename(file_path)}
    )

    # 等待檔案處理完成
    while uploaded_file.state.name == "PROCESSING":
        print("檔案處理中...")
        time.sleep(1)
        uploaded_file = client.files.get(name=uploaded_file.name)

    if uploaded_file.state.name == "FAILED":
        raise Exception("檔案上傳處理失敗")

    return uploaded_file

def query_with_rag(user_id, question):
    """使用 RAG 模式回答問題"""
    session = get_user_session(user_id)
    uploaded_file = session["uploaded_file"]

    if uploaded_file:
        # 有上傳檔案，使用 RAG 模式
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(
                            file_uri=uploaded_file.uri,
                            mime_type=uploaded_file.mime_type
                        ),
                        types.Part.from_text(text=f"請根據上述提供的檔案內容，用繁體中文回答這個問題：{question}")
                    ]
                )
            ]
        )
        return response.text
    else:
        # 沒有上傳檔案，使用一般多輪對話
        response = session["chat"].send_message(message=question)
        return response.text

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature.")
        abort(400)

    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    """處理文字訊息"""
    text = event.message.text
    user_id = event.source.user_id

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            try:
                # 使用 RAG 或一般對話
                reply_text = query_with_rag(user_id, prompt)

                # 檢查是否有上傳檔案，加上提示
                session = get_user_session(user_id)
                if session["uploaded_file"]:
                    reply_text = f"📄 [RAG 模式]\n\n{reply_text}"

                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=reply_text)]
                    )
                )
            except Exception as e:
                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=f"❌ 發生錯誤：{str(e)}")]
                    )
                )
        elif text == "清除文件":
            # 清除使用者上傳的檔案
            session = get_user_session(user_id)
            session["uploaded_file"] = None
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="✅ 已清除上傳的文件，恢復一般對話模式。")]
                )
            )
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="請輸入「AI 問題」來開始對話\n或上傳 TXT/PDF 檔案啟用 RAG 模式")]
                )
            )

@handler.add(MessageEvent, message=FileMessageContent)
def handle_file_message(event):
    """處理使用者上傳的檔案"""
    user_id = event.source.user_id
    file_name = event.message.file_name

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 檢查檔案類型
        if not (file_name.endswith('.txt') or file_name.endswith('.pdf')):
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="⚠️ 目前只支援 TXT 或 PDF 檔案")]
                )
            )
            return

        try:
            # 下載檔案
            file_path = download_line_file(event.message.id, file_name)
            print(f"檔案已下載：{file_path}")

            # 上傳到 Gemini
            uploaded_file = upload_file_to_gemini(file_path)
            print(f"檔案已上傳到 Gemini：{uploaded_file.uri}")

            # 儲存到使用者 session
            session = get_user_session(user_id)
            session["uploaded_file"] = uploaded_file

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"✅ 檔案「{file_name}」上傳成功！\n\n現在您可以輸入「AI 問題」來詢問關於這份文件的問題。\n\n輸入「清除文件」可恢復一般對話模式。")]
                )
            )
        except Exception as e:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"❌ 檔案處理失敗：{str(e)}")]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:__main__:Request body: {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[]}
INFO:werkzeug:127.0.0.1 - - [08/Jan/2026 15:18:54] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[]}


INFO:__main__:Request body: {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[{"type":"message","message":{"type":"text","id":"595682432011993295","quoteToken":"TGibA3FEVPyyrpBvt8fATP4X0jylH3QCGAsEi7jU0_2UrDv82XyYAwSTYa1d1KdFyh48DsvHL_pmR4470wkyyZ-HZNbB4sq6W6WzkjChhwl43o8QyTGCQrFOrJU76OYIKpa-DV34jubyRyylp6xZDQ","markAsReadToken":"FZAy-8l0C92LyRFE8QgepE-71kYKNublVkxMwJ4T6nXB1Ea9ArQp6_kGGQ4OGiTqz_K-u5eTKaNmKifr24R6LCmWAteugN918xWqiQtOSc_iMZqymYs3M81GT4Id4Y1pq7X1MtBZqgAxR0wdjjKXZHMruD_OIDx30pdaVjvzJt79OGUCydqkwJHklENOgbsoDI5Dr-trguUBnotvrgK47A","text":"AI 大俠愛吃甚麼？"},"webhookEventId":"01KEF3612BFWZ56H0PSTFM988C","deliveryContext":{"isRedelivery":false},"timestamp":1767885701710,"source":{"type":"user","userId":"Ue11b87b78100ff0db23f5a92428c2878"},"replyToken":"76c590fd284e4e28b30c36dbfed20160","mode":"active"}]}


BODY:  {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[{"type":"message","message":{"type":"text","id":"595682432011993295","quoteToken":"TGibA3FEVPyyrpBvt8fATP4X0jylH3QCGAsEi7jU0_2UrDv82XyYAwSTYa1d1KdFyh48DsvHL_pmR4470wkyyZ-HZNbB4sq6W6WzkjChhwl43o8QyTGCQrFOrJU76OYIKpa-DV34jubyRyylp6xZDQ","markAsReadToken":"FZAy-8l0C92LyRFE8QgepE-71kYKNublVkxMwJ4T6nXB1Ea9ArQp6_kGGQ4OGiTqz_K-u5eTKaNmKifr24R6LCmWAteugN918xWqiQtOSc_iMZqymYs3M81GT4Id4Y1pq7X1MtBZqgAxR0wdjjKXZHMruD_OIDx30pdaVjvzJt79OGUCydqkwJHklENOgbsoDI5Dr-trguUBnotvrgK47A","text":"AI 大俠愛吃甚麼？"},"webhookEventId":"01KEF3612BFWZ56H0PSTFM988C","deliveryContext":{"isRedelivery":false},"timestamp":1767885701710,"source":{"type":"user","userId":"Ue11b87b78100ff0db23f5a92428c2878"},"replyToken":"76c590fd284e4e28b30c36dbfed20160","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [08/Jan/2026 15:21:52] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[{"type":"message","message":{"type":"file","id":"595683568684892747","markAsReadToken":"LNUpFME0IynuTmNkAEd5Wz14rUMqshtmXgy4ejxB5pFuOXSqmabiHRwaOd04KuIbS_hgYApSMpQKQyjW0_kPeSJDVjntoalYexRqsTkxVfwsCnATLzcqGM-F7wGWP7yWF1TBj4ptL3dBDtgVTktzWqH7AmqznhRnoOvsmnE_GTHWZiZWjhL9Rd1FB1EcQ4B9be6bulOKT8wB32MaK-dfwg","fileName":"faq.txt","fileSize":27,"contentProvider":{"type":"line"}},"webhookEventId":"01KEF3TPDSARXD0EC109MVSV7B","deliveryContext":{"isRedelivery":false},"timestamp":1767886379175,"source":{"type":"user","userId":"Ue11b87b78100ff0db23f5a92428c2878"},"replyToken":"dcd0af62e08d4bd4b9de16a7b05a99c3","mode":"active"}]}


BODY:  {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[{"type":"message","message":{"type":"file","id":"595683568684892747","markAsReadToken":"LNUpFME0IynuTmNkAEd5Wz14rUMqshtmXgy4ejxB5pFuOXSqmabiHRwaOd04KuIbS_hgYApSMpQKQyjW0_kPeSJDVjntoalYexRqsTkxVfwsCnATLzcqGM-F7wGWP7yWF1TBj4ptL3dBDtgVTktzWqH7AmqznhRnoOvsmnE_GTHWZiZWjhL9Rd1FB1EcQ4B9be6bulOKT8wB32MaK-dfwg","fileName":"faq.txt","fileSize":27,"contentProvider":{"type":"line"}},"webhookEventId":"01KEF3TPDSARXD0EC109MVSV7B","deliveryContext":{"isRedelivery":false},"timestamp":1767886379175,"source":{"type":"user","userId":"Ue11b87b78100ff0db23f5a92428c2878"},"replyToken":"dcd0af62e08d4bd4b9de16a7b05a99c3","mode":"active"}]}
檔案已下載：/content/uploaded_files/faq.txt
檔案已上傳到 Gemini：https://generativelanguage.googleapis.com/v1beta/files/0lep4oxmpgwd


INFO:werkzeug:127.0.0.1 - - [08/Jan/2026 15:33:02] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[{"type":"message","message":{"type":"text","id":"595683600377840215","quoteToken":"DL8tE2XMhGq5_uQowiqMJaw0_8P6-pa1FjhHetUWTZ9_zHWd8Jz757AgGGJgX5EL7K4vrSFdq6JfCWpf0Qzi6U0ILjEF-Nnvd1RLbnBedK0GgeL6F2eaOFDmAihNSloRuD8GedLZHy3oytweWGgSrA","markAsReadToken":"00HdHzrVhCt5i6DylbnBtS4g2UaY0i0ckvjB0TZkB5kf9tQiYSaWmHwbEuG7LXuNEzsMKOJn1HEK5tTOxWRc3OHZnFcSeVIxj2F52uaXt5Z3xwJMh3n51q-SJYSf2nEoph1RyVrglovQDQFEglYH-zcjzdmPwxVpBBP3ztW6Zpme2Khm9Sj6dniE1i04wxBAKphc_uqH-RbSjxCNUo_KMA","text":"AI 大俠愛吃甚麼？"},"webhookEventId":"01KEF3V93WDZ5Y1RX4K9HS7BC1","deliveryContext":{"isRedelivery":false},"timestamp":1767886398115,"source":{"type":"user","userId":"Ue11b87b78100ff0db23f5a92428c2878"},"replyToken":"ed60e38d2d46414f8efa5f43552cea8e","mode":"active"}]}


BODY:  {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[{"type":"message","message":{"type":"text","id":"595683600377840215","quoteToken":"DL8tE2XMhGq5_uQowiqMJaw0_8P6-pa1FjhHetUWTZ9_zHWd8Jz757AgGGJgX5EL7K4vrSFdq6JfCWpf0Qzi6U0ILjEF-Nnvd1RLbnBedK0GgeL6F2eaOFDmAihNSloRuD8GedLZHy3oytweWGgSrA","markAsReadToken":"00HdHzrVhCt5i6DylbnBtS4g2UaY0i0ckvjB0TZkB5kf9tQiYSaWmHwbEuG7LXuNEzsMKOJn1HEK5tTOxWRc3OHZnFcSeVIxj2F52uaXt5Z3xwJMh3n51q-SJYSf2nEoph1RyVrglovQDQFEglYH-zcjzdmPwxVpBBP3ztW6Zpme2Khm9Sj6dniE1i04wxBAKphc_uqH-RbSjxCNUo_KMA","text":"AI 大俠愛吃甚麼？"},"webhookEventId":"01KEF3V93WDZ5Y1RX4K9HS7BC1","deliveryContext":{"isRedelivery":false},"timestamp":1767886398115,"source":{"type":"user","userId":"Ue11b87b78100ff0db23f5a92428c2878"},"replyToken":"ed60e38d2d46414f8efa5f43552cea8e","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [08/Jan/2026 15:33:22] "POST / HTTP/1.1" 200 -
